# 07 Website Export

In [1]:
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
RESULTS = Path("../results")
WEB = Path("../website")
(WEB / "js" / "generated").mkdir(parents=True, exist_ok=True)
(WEB / "tests").mkdir(exist_ok=True)

cfg = json.loads((MODELS / "model_config.json").read_text())
FEATURES, TAU = cfg["features"], cfg["threshold"]
champion = lgb.Booster(model_file=str(MODELS / "lgbm_champion.txt"))
read = lambda name: json.loads((RESULTS / name).read_text())

## Site Data

In [2]:
site = {"data": read("data_summary.json"), "raw": read("raw_summary.json"), "eda": read("eda_summary.json"), **read("evaluation.json"),
        "val_metrics": pd.read_csv(RESULTS / "validation_metrics.csv").to_dict(orient="records"),
        "tuning": cfg["tuning"], "optuna": cfg["optuna_trials"]}
(WEB / "js/generated/site_data.js").write_text("// Generated by notebooks/07_website_export.ipynb\nwindow.SITE_DATA = "
                                              + json.dumps(site, default=float, separators=(",", ":")) + ";\n", encoding="utf-8")
print(f"site_data.js {(WEB / 'js/generated/site_data.js').stat().st_size / 1e3:.0f} kB")

site_data.js 81 kB


## Champion Model to JavaScript

In [3]:
MISSING = {"None": 0, "Zero": 1, "NaN": 2}

def flatten(tree):
    nodes = {"f": [], "t": [], "cat": [], "dl": [], "mt": [], "l": [], "r": [], "v": []}
    leaves = []
    def walk(n):
        if "leaf_value" in n:
            leaves.append(n["leaf_value"]); return -len(leaves)  # leaf k is stored as -(k+1)
        i = len(nodes["f"])
        for k in nodes: nodes[k].append(None)
        is_cat = n["decision_type"] == "=="
        nodes["f"][i] = n["split_feature"]; nodes["cat"][i] = int(is_cat)
        nodes["t"][i] = [int(x) for x in str(n["threshold"]).split("||")] if is_cat else n["threshold"]
        nodes["dl"][i] = int(n["default_left"]); nodes["mt"][i] = MISSING[n["missing_type"]]
        nodes["v"][i] = n["internal_value"]
        nodes["l"][i] = walk(n["left_child"]); nodes["r"][i] = walk(n["right_child"])
        return i
    walk(tree["tree_structure"])
    return {**nodes, "leaf": leaves}

dump = champion.dump_model()
model_js = {"features": dump["feature_names"], "trees": [flatten(t) for t in dump["tree_info"]]}
print(len(model_js["trees"]), "trees")

861 trees


## Predictor Form to Model Features

In [4]:
X_train = pd.read_parquet(PROCESSED / "X_train.parquet")[FEATURES]
y_train = pd.read_parquet(PROCESSED / "y_train.parquet").is_churn.values
logged = X_train[X_train.has_logs == 1]
med = lambda s: float(np.nanmedian(s))
listen_med = {"secs_per_song": med(logged.secs_per_day / logged.songs_per_day.replace(0, np.nan)),
              "unique_song_ratio": med(logged.unique_song_ratio), "complete_ratio": med(logged.complete_ratio),
              "daily_secs_cv": med(logged.daily_secs_cv)}
FILL = ["tx_count", "auto_renew_ratio", "discount_count", "n_payment_methods", "cancel_count"]
group_fill = {f"{int(a)}_{int(c)}": {k: med(d[k]) for k in FILL}
              for (a, c), d in X_train[X_train.has_tx == 1].groupby(["last_auto_renew", "last_is_cancel"])}

def form_to_features(p):
    # a cancellation always comes with auto-renew on in the data, so an unseen combination uses the cancel group
    g = group_fill.get(f"{p['last_is_auto_renew']}_{p['last_is_cancel']}") or group_fill[f"1_{p['last_is_cancel']}"]
    hours, active = max(0.0, p["total_secs_hours"]), int(min(max(p["active_days_count"], 0), 31))
    r = {"city": p["city"], "registered_via": p["registered_via"], "gender": p["gender_encoded"],
         "age": p["bd_cleaned"] if 10 <= p["bd_cleaned"] <= 90 else np.nan, "tenure_days": p["tenure_days"], "has_member": 1,
         "has_tx": 1, "tx_count": g["tx_count"], "total_paid": g["tx_count"] * p["last_amount_paid"], "avg_plan_days": p["last_plan_days"],
         "auto_renew_ratio": g["auto_renew_ratio"], "cancel_count": max(g["cancel_count"], p["last_is_cancel"]),
         "discount_count": g["discount_count"], "n_payment_methods": g["n_payment_methods"],
         "last_payment_method": p["last_payment_method"], "last_plan_days": p["last_plan_days"], "last_amount_paid": p["last_amount_paid"],
         "last_auto_renew": p["last_is_auto_renew"], "last_is_cancel": p["last_is_cancel"],
         "days_since_last_tx": max(0, p["last_plan_days"] - p["days_until_expiration"]),  # last payment = expiry minus plan length
         "days_until_expiration": p["days_until_expiration"]}
    if hours > 0 and active > 0:
        spd = hours * 3600 / active
        dsla = int(min(max(p["days_since_last_active"], 0), 30))
        share = float(min(max(p["recent_activity_ratio"], 0), 1))
        r.update({"has_logs": 1, "active_days": active, "total_hours": hours, "secs_per_day": spd,
                  "songs_per_day": spd / listen_med["secs_per_song"], "unique_song_ratio": listen_med["unique_song_ratio"],
                  "skip_ratio": p["skip_ratio"], "complete_ratio": min(listen_med["complete_ratio"], 1 - p["skip_ratio"]),
                  "last7_share": share, "active_days_7d": 0 if dsla >= 7 else max(1, min(7, active, round(active * share))),
                  "days_since_last_active": dsla, "daily_secs_cv": listen_med["daily_secs_cv"]})
    else:
        r.update({k: 0 for k in ["active_days", "total_hours", "secs_per_day", "songs_per_day", "unique_song_ratio", "skip_ratio",
                                 "complete_ratio", "last7_share", "active_days_7d", "daily_secs_cv", "has_logs"]})
        r["days_since_last_active"] = 31
    return pd.DataFrame([r])[FEATURES].astype(float)

# observed churn rate of test subscribers in each score band (Wilson 95% interval), shown next to a prediction
X_test = pd.read_parquet(PROCESSED / "X_test.parquet")[FEATURES]
y_test = pd.read_parquet(PROCESSED / "y_test.parquet").is_churn.values
p_test = champion.predict(X_test)
edges = [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0001]
score_bins = []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (p_test >= lo) & (p_test < hi); n, k = int(m.sum()), int(y_test[m].sum())
    if n == 0: continue
    ph, z = k / n, 1.96
    c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    score_bins.append({"lo": lo, "hi": min(hi, 1.0), "n": n, "rate": round(ph, 4), "ci": [round(max(0, c - h), 4), round(min(1, c + h), 4)]})

meta = {"threshold": TAU, "listen_med": listen_med, "group_fill": group_fill, "score_bins": score_bins, "base_rate": float(y_train.mean())}
(WEB / "js/generated/model_lgbm.js").write_text("// Generated by notebooks/07_website_export.ipynb from models/lgbm_champion.txt\n"
                                               "window.LGBM_MODEL = " + json.dumps(model_js, separators=(",", ":")) + ";\n"
                                               "window.PREDICTOR_META = " + json.dumps(meta, separators=(",", ":")) + ";\n", encoding="utf-8")
print(f"model_lgbm.js {(WEB / 'js/generated/model_lgbm.js').stat().st_size / 1e6:.2f} MB")

model_lgbm.js 2.86 MB


## Parity Fixtures

In [5]:
rows = X_test.sample(300, random_state=1)
rng = np.random.default_rng(3)
forms = [{"last_is_auto_renew": int(rng.integers(0, 2)), "last_is_cancel": int(rng.integers(0, 2)),
          "last_plan_days": int(rng.choice([7, 30, 90, 180, 410])), "last_amount_paid": float(rng.choice([0, 99, 149, 180, 450, 1788])),
          "days_until_expiration": int(rng.integers(-30, 400)), "last_payment_method": int(rng.choice([41, 38, 36, 39, 40])),
          "total_secs_hours": float(np.round(rng.exponential(25), 1)) * int(rng.random() > 0.2), "active_days_count": int(rng.integers(0, 31)),
          "days_since_last_active": int(rng.integers(0, 31)), "recent_activity_ratio": float(np.round(rng.random(), 2)),
          "skip_ratio": float(np.round(rng.random() * 0.6, 2)), "bd_cleaned": int(rng.choice([0, 17, 25, 34, 60])),
          "gender_encoded": int(rng.integers(0, 3)), "tenure_days": int(rng.integers(0, 4000)), "city": 1, "registered_via": 7}
         for _ in range(100)]
fixtures = {"rows": rows.astype(object).where(rows.notna(), None).values.tolist(), "row_probs": champion.predict(rows).tolist(),
            "forms": forms, "form_probs": [float(champion.predict(form_to_features(f))[0]) for f in forms]}
(WEB / "tests/parity_fixtures.json").write_text(json.dumps(fixtures))
print(f"saved {len(rows)} test rows and {len(forms)} form inputs for the parity test")

saved 300 test rows and 100 form inputs for the parity test


## Kaggle Test Predictions

In [6]:
kaggle = pd.read_parquet(PROCESSED / "features_kaggle_test.parquet")
pd.DataFrame({"msno": kaggle.msno, "is_churn": champion.predict(kaggle[FEATURES]).round(6)}).to_csv(RESULTS / "submission.csv", index=False)
print(f"submission.csv: {len(kaggle):,} subscribers")

submission.csv: 24,274 subscribers
